In [ ]:
import os
from paths import CLINVAR_CSV, CONFIG, OUTPUT_DIR, SCORES_DIR

import pandas as pd
from aigct.container import VEBenchmarkContainer
container = VEBenchmarkContainer(CONFIG)

query_mgr = container.query_mgr


In [ ]:
tasks_df = query_mgr.get_tasks()

In [ ]:
from aigct.model import VEQueryCriteria
qry1 = VEQueryCriteria(filter_names=['CHD_case','CHD_control1','CHD_control2','CHD_control3','CHD_control4'])

In [ ]:
clinvar = pd.read_csv(CLINVAR_CSV)
qry1 = VEQueryCriteria(variant_ids = clinvar , include_variant_ids= False)

In [ ]:
CLINV = query_mgr.get_variants_by_task('CHD',qry1)
print(len(CLINV[CLINV['BINARY_LABEL'] == 0]), len(CLINV[CLINV['BINARY_LABEL'] == 1]))

In [ ]:
from aigct.container import VEBenchmarkContainer

container = VEBenchmarkContainer(CONFIG)

query_mgr = container.query_mgr
CHD = query_mgr.get_variants_by_task('CHD', qry1)
CHD.to_csv(os.path.join(OUTPUT_DIR, "CHD_pri_study2_EVE.csv"))

In [ ]:
CHD_MAVEN_EVE = pd.read_csv(os.path.join(SCORES_DIR, "CHD_AIGCT.csv"))
CHD_pri_EVE = pd.read_csv(os.path.join(SCORES_DIR, "CHD_pri_EVE.csv"))
CHD_EVE = pd.concat([CHD_MAVEN_EVE, CHD_pri_EVE])

In [ ]:
#MAVEN
metrics = container.analyzer.compute_metrics(
    "CHD", CHD_MAVEN_EVE,'EVE',variant_effect_sources= ['MAVEN', 'MAVENAVG', 'ALPHAM', 'REVEL', 'ESM1B'], vep_min_overlap_percent=1, variant_vep_retention_percent=100)

In [ ]:
#PRIMATE AI
metrics = container.analyzer.compute_metrics(
    "CHD", CHD_EVE,'EVE',variant_effect_sources= ['PROVEAN', 'MAVENAVG', 'POLYP2HVAR', 'PRIMAI',  'REVEL', 'DEOGEN2', 'CADD', 'VEST4', 'SIFT', 'BAYESDNAF', 'MUTTASTE', 'LISTS2', 'MCAP', 'FATHMMXF', 'DANN'], vep_min_overlap_percent=1, variant_vep_retention_percent=100)

In [ ]:
#ALL
metrics = container.analyzer.compute_metrics(
    "CHD",
    #variant_query_criteria = qry1,  
    vep_min_overlap_percent=90, 
    variant_vep_retention_percent=100,)

In [ ]:
container.reporter.write_summary(metrics)

In [ ]:
container.plotter.plot_results(metrics)


In [ ]:
vep_stats_df = query_mgr.get_variant_effect_source_stats("CANCER",
    ["ALPHAM", "REVEL", "EVE"])

In [ ]:
vep_stats_df